# 03 — Baseline CNN trained from scratch (Tasks 3 and 4)

A small convolutional network (`src/models.py::SimpleCNN`) trained from random initialisation at 96 px on fold 0
(train = folds 1–4, validation = fold 0). Two runs:

| run | augmentation | label smoothing |
|---|---|---|
| `cnn_noaug` | none (preprocessing only) | 0.0 |
| `cnn_shift` | `shift` (framing / resolution augmentation, see notebook 02) | 0.1 |

Each epoch logs training loss / accuracy and validation loss / accuracy on **val_clean** (plain preprocessing) and
**val_shift** (the same validation images passed through ShiftSim). The checkpoint with the best val_shift accuracy
is kept. Training takes roughly 2 minutes per epoch on the M1 Pro (20 epochs per run).

The notebook ends by writing the **first Kaggle submission** from the better of the two runs.

Runs are resumable: if the kernel stops, re-running the notebook continues from the last finished epoch, and
finished runs are not retrained (set `FORCE = True` to retrain).

In [1]:
RUNS = "cnn_noaug,cnn_shift"   # comma-separated preset names from src/config.py
FOLD = 0                       # validation fold
FORCE = False                  # True = delete the checkpoints of these runs and train again
SUBMIT_RUN = "auto"            # run used for the submission; "auto" = highest val_shift accuracy

In [2]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd()))  # the notebooks live in the repo root, next to src/

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

sns.set_theme(style="whitegrid")
pd.set_option("display.precision", 4)
pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 200)

import torch
from sklearn.metrics import confusion_matrix

from src.config import CLASSES, EXPERIMENTS_CSV, PRESETS, SUBMISSIONS_CSV, get_device, make_config, make_dirs
from src.data import load_test_df
from src.engine import fit, get_preds
from src.models import count_params, create_model
from src.plots import plot_confusion, plot_history, savefig
from src.utils import accuracy, softmax, write_submission

make_dirs()
print("device:", get_device())

runs = [r.strip() for r in RUNS.split(',') if r.strip()]

device: mps


In [3]:
def experiments_for(runs, folds=None):
    """Rows of results/experiments.csv for the given runs (and folds)."""
    ex = pd.read_csv(EXPERIMENTS_CSV)
    ex = ex[ex["run"].isin(runs)]
    if folds is not None:
        ex = ex[ex["fold"].isin(folds)]
    return ex.reset_index(drop=True)


def per_class_accuracy(run, fold):
    """Accuracy per true class on val_clean and val_shift for one fold."""
    out = {}
    for split in ["val_clean", "val_shift"]:
        _, logits, labels = get_preds(run, fold, split)
        pred = logits.argmax(1)
        out[split] = [(pred[labels == k] == k).mean() for k in range(len(CLASSES))]
    return pd.DataFrame(out, index=CLASSES)

## 1. Configuration of the runs

In [4]:
pd.DataFrame({r: make_config(r).to_dict() for r in runs})

,cnn_noaug,cnn_shift
run_name,cnn_noaug,cnn_shift
model_name,simple_cnn,simple_cnn
pretrained,False,False
img_size,96,96
resize_mode,squash,squash
aug,none,shift
crop_scale_min,0.6,0.6
crop_ratio,"(0.6, 1.67)","(0.6, 1.67)"
lowres_p,0.3,0.3
epochs,20,20


## 2. Model architecture

In [ ]:
cfg0 = make_config(runs[0])
model = create_model(cfg0)
print(model)
x = torch.zeros(2, 3, cfg0.img_size, cfg0.img_size)
print("\ninput", tuple(x.shape), "-> features", tuple(model.features(x).shape), "-> logits", tuple(model(x).shape))
print(f"trainable parameters: {count_params(model):,}")

SimpleCNN(
  (features): Sequential(
    (0): Sequential(
      (0): Conv2d(3, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
      (2): ReLU(inplace=True)
      (3): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (4): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
      (5): ReLU(inplace=True)
      (6): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    )
    (1): Sequential(
      (0): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
      (2): ReLU(inplace=True)
      (3): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (4): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, tr

: 

## 3. Training

In [ ]:
histories = {}
for run in runs:
    histories[run] = fit(make_config(run), fold=FOLD, force=FORCE)

[cnn_noaug fold 0] device=mps  train=50914  val=12732  steps/epoch=397


train:   0%|          | 0/397 [00:10<?, ?it/s]

val_clean:   0%|          | 0/50 [00:10<?, ?it/s]

val_shift:   0%|          | 0/50 [00:10<?, ?it/s]

epoch  1/20  train 0.8013/0.7208  clean 0.4852/0.8273  shift 0.5601/0.8014  124s
  ETA for the remaining 19 epochs: 39 min


train:   0%|          | 0/397 [00:00<?, ?it/s]

val_clean:   0%|          | 0/50 [00:00<?, ?it/s]

val_shift:   0%|          | 0/50 [00:00<?, ?it/s]

epoch  2/20  train 0.1667/0.9442  clean 0.2705/0.9087  shift 0.3612/0.8772  92s


train:   0%|          | 0/397 [00:00<?, ?it/s]

## 4. Learning curves

In [ ]:
for run, hist in histories.items():
    plot_history(hist, title=f"{run} (fold {FOLD})", name=f"03_curves_{run}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
for run, hist in histories.items():
    axes[0].plot(hist["epoch"], hist["train_acc"] - hist["val_clean_acc"], marker="o", label=run)
    axes[1].plot(hist["epoch"], hist["val_clean_acc"] - hist["val_shift_acc"], marker="o", label=run)
axes[0].set_title("train accuracy − val_clean accuracy")
axes[1].set_title("val_clean accuracy − val_shift accuracy")
for ax in axes:
    ax.axhline(0, color="grey", lw=0.8)
    ax.set_xlabel("epoch")
    ax.legend()
plt.tight_layout()
savefig("03_gaps")
plt.show()

> **Observations (to be written by me):**

## 5. Results at the best epoch (selected on val_shift)

In [ ]:
ex = experiments_for(runs, [FOLD])
display(ex[["run", "best_epoch", "epochs_run", "train_acc", "val_clean_acc", "val_shift_acc",
            "val_clean_loss", "val_shift_loss", "minutes"]])
display(pd.concat({run: per_class_accuracy(run, FOLD) for run in runs}, axis=1).round(4))

In [ ]:
for run in runs:
    fig, axes = plt.subplots(1, 2, figsize=(15, 6))
    for ax, split in zip(axes, ["val_clean", "val_shift"]):
        _, logits, labels = get_preds(run, FOLD, split)
        plot_confusion(confusion_matrix(labels, logits.argmax(1), labels=range(len(CLASSES))), CLASSES,
                       title=f"{run} {split}", ax=ax)
    plt.tight_layout()
    savefig(f"03_confusion_{run}")
    plt.show()

> **Observations (to be written by me):**

## 6. First submission

Test predictions of the selected run (fold-0 model, no test-time augmentation) → `submissions/<run>_fold0.csv`.
The file is checked against `sample_submission.csv` (29,443 rows, identical ids, valid class names) and logged in
`results/submissions.csv` with its validation scores. After uploading it to Kaggle, type the public leaderboard
score into the `lb_public` column of `results/submissions.csv`.

In [ ]:
if SUBMIT_RUN == "auto":
    SUBMIT_RUN = ex.sort_values("val_shift_acc", ascending=False)["run"].iloc[0]
row = ex[ex["run"] == SUBMIT_RUN].iloc[0]
test_df = load_test_df()
ids, logits, _ = get_preds(SUBMIT_RUN, FOLD, "test")
assert (ids == test_df["image_id"].values).all()
write_submission(test_df, softmax(logits), f"{SUBMIT_RUN}_fold{FOLD}", runs=SUBMIT_RUN,
                 val_clean=row["val_clean_acc"], val_shift=row["val_shift_acc"], notes="baseline CNN, no TTA")

In [ ]:
pd.read_csv(SUBMISSIONS_CSV)